<img src="https://raw.githubusercontent.com/telmomm/CERTA/main/docs/assets/logo/certa-logo.png" alt="CERTA" width="220">

# T0.3 · Reviewer sheet

*The screening spreadsheet sent to the reviewers, the sheets they return and the agreement between them*

Writes the spreadsheet that the screening reviewers fill in, following `../collaborators/screening-instructions.md`; brings the files they return to that same format; and measures the agreement of each reviewer with the others and with the main reviewer (`screening.csv`).

- **Input:** `../T0.2-search-and-deduplication/corpus.csv`; `screening.csv`; the files returned by the reviewers in `reviewers/raw/`, as `.xlsx` (the sheet) or `.md` (a Markdown table with `id`, `decision`, `reason` and optionally `notes`).
- **Output:** `screening-sheet.xlsx` (the sheet to send); `reviewers/screening-sheet-<name>.xlsx` (each returned file in the format of the sheet); `reviewer_decisions.csv` (every decision of every reviewer, one row per record and reviewer); `reviewer_agreement.csv` (agreement between each pair).
- **To re-run:** run all cells. The returned files are only read; the files derived from them are written again.
- **Environment:** the repository `.venv` (see `CONTRIBUTING.md`).

In [1]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import re
from itertools import combinations
import numpy as np
import pandas as pd
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation

pd.set_option("display.max_colwidth", 90)
CERTA = {"navy": "#0B2D4A", "teal": "#00BFA5", "blue": "#00AEEF", "sky": "#A7E3F7", "ice": "#E6EEF6"}

class table:
    """Shows a DataFrame with the CERTA colours in the notebook and as a LaTeX table in the PDF export.
    `bar` is a numeric column drawn as bars."""
    def __init__(self, df, bar=None):
        self.df, self.bar = df, bar
    def _repr_html_(self):
        styler = self.df.style.hide(axis="index").set_table_styles([
            {"selector": "th", "props": f"background-color: {CERTA['navy']}; color: white; text-align: left;"},
            {"selector": "tbody tr:nth-child(even)", "props": f"background-color: {CERTA['ice']};"},
        ])
        if self.bar is not None:
            styler = styler.bar(subset=[self.bar], color=CERTA["teal"], vmin=0)
        return styler.to_html()
    def _repr_latex_(self):
        return self.df.style.hide(axis="index").format(precision=3).to_latex(hrules=True)
    def __repr__(self):
        return self.df.to_string(index=False)

CORPUS = Path("../T0.2-search-and-deduplication/corpus.csv")
SCREENING = Path("screening.csv")
SHEET = Path("screening-sheet.xlsx")
RAW = Path("reviewers/raw")             # files as returned by each reviewer, never edited: <name>.xlsx or <name>.md
RETURNED = Path("reviewers")            # the same, in the format of the sheet: screening-sheet-<name>.xlsx
DECISIONS_ALL = Path("reviewer_decisions.csv")
AGREEMENT = Path("reviewer_agreement.csv")
SEED = 2026                             # fixed order of the records in the sheet

READ = ["id", "title", "year", "source", "abstract"]
ANSWER = ["decision", "reason", "notes"]
DECISIONS = ["include", "exclude", "doubt"]
REASONS = ["E1", "E2", "E3", "E4", "E5"]

## 1. Records of the sheet

Every record of the corpus, with only what is needed to decide: no databases, blocks, keywords, reading hint or decisions, which could bias the reviewer. The order is random with a fixed seed, so that it does not follow the reading order of the main reviewer.

In [2]:
corpus = pd.read_csv(CORPUS, dtype=str).fillna("")
records = corpus[READ].sample(frac=1, random_state=SEED).reset_index(drop=True)
for column in ANSWER:
    records[column] = ""
print("records:", len(records), "| without abstract:", (records["abstract"].str.len() < 50).sum())
table(records.head(3)[["id", "year", "title"]])

records: 2734 | without abstract: 14


id,year,title
R1911,1998,A Neural-Based Analog IC Parametric Fault Diagnostic System with Discriminant Simulation Sampling
R0663,2001,Testing IP cores with pseudo exhaustive test sets
R0478,1999,Accelerating test data processing


## 2. Writing the sheet

One sheet, `Screening`, with the columns to read shaded and the three answer columns. `decision` and `reason` only accept the codes of the criteria sheet, from a drop-down list. A second sheet, `Codes`, repeats their meaning. The same writer is used for the blank sheet and for the returned files of section 3.

In [3]:
WIDTH = {"id": 9, "title": 45, "year": 7, "source": 28, "abstract": 95, "decision": 12, "reason": 9, "notes": 30}
CODES = [
    ("decision", "include", "Meets the three questions of the criteria sheet"),
    ("decision", "exclude", "Clearly fails one of them; write the reason"),
    ("decision", "doubt", "The abstract is not enough to decide; settled with the full text"),
    ("reason", "E1", "Digital only"),
    ("reason", "E2", "Diagnosis without selection"),
    ("reason", "E3", "Other domain"),
    ("reason", "E4", "Test technique or structure without selection or coverage"),
    ("reason", "E5", "Not a primary study"),
    ("notes", "review", "The record is a review paper"),
    ("notes", "own", "The record is by you or your group: leave decision empty"),
]

def write_sheet(rows, path):
    """Writes the screening sheet with the records of `rows` (columns READ + ANSWER), in their order."""
    navy, ice, white = CERTA["navy"][1:], CERTA["ice"][1:], "FFFFFF"
    head_font, head_fill = Font(bold=True, color=white), PatternFill("solid", fgColor=navy)
    read_fill, top = PatternFill("solid", fgColor=ice), Alignment(vertical="top", wrap_text=True)
    wb = Workbook()
    ws = wb.active
    ws.title = "Screening"
    ws.append(READ + ANSWER)
    for row in rows[READ + ANSWER].itertuples(index=False):
        values = list(row)
        values[READ.index("year")] = int(row.year) if row.year.isdigit() else row.year
        ws.append(values)
    last = len(rows) + 1
    for c, name in enumerate(READ + ANSWER, start=1):
        ws.column_dimensions[ws.cell(1, c).column_letter].width = WIDTH[name]
        ws.cell(1, c).font, ws.cell(1, c).fill = head_font, head_fill
        for r in range(2, last + 1):
            ws.cell(r, c).alignment = top
            if name in READ:
                ws.cell(r, c).fill = read_fill
    # Row height from the length of the abstract, since Excel does not always fit wrapped text by itself
    for r, text in enumerate(rows["abstract"], start=2):
        ws.row_dimensions[r].height = min(409, 15 * max(2, int(np.ceil(len(text) / (WIDTH["abstract"] * 1.15)))))
    for options, column in [(DECISIONS, "decision"), (REASONS, "reason")]:
        letter = ws.cell(1, (READ + ANSWER).index(column) + 1).column_letter
        rule = DataValidation(type="list", formula1='"' + ",".join(options) + '"', allow_blank=True,
                              showErrorMessage=True, errorTitle="Not a valid code",
                              error="Choose one of: " + ", ".join(options))
        rule.add(f"{letter}2:{letter}{last}")
        ws.add_data_validation(rule)
    ws.freeze_panes = "B2"
    ws.auto_filter.ref = f"A1:{ws.cell(1, len(READ + ANSWER)).column_letter}{last}"
    codes = wb.create_sheet("Codes")
    codes.append(["column", "code", "meaning"])
    for row in CODES:
        codes.append(row)
    for c, width in zip("ABC", (10, 10, 70)):
        codes.column_dimensions[c].width = width
        codes[f"{c}1"].font, codes[f"{c}1"].fill = head_font, head_fill
    wb.properties.creator = "CERTA"
    wb.properties.created = wb.properties.modified = datetime(2026, 10, 17)
    wb.save(path)

write_sheet(records, SHEET)
print("written", SHEET, "with", len(records), "records")

written screening-sheet.xlsx with 2734 records


## 3. Returned files

Each reviewer returns either the sheet itself or a Markdown table. Both are read, cleaned in the same way and written again as `reviewers/screening-sheet-<name>.xlsx`, with the records in the order of the sheet that was sent:

- codes are lower-cased (`decision`) or upper-cased (`reason`), and anything that is not a valid code is left blank and counted as invalid;
- a record whose `id` is not in the corpus (for example a row cut short) is dropped and counted;
- a record that appears more than once keeps its first answer; if its answers differ, it is left undecided and counted as conflicting;
- a record missing from the file stays undecided.

In [4]:
def read_markdown(path):
    """The first Markdown table of a file, as a DataFrame with lower-case column names."""
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line.startswith("|") or re.fullmatch(r"\|[\s:|-]+\|?", line):
            continue
        rows.append([c.strip() for c in line.strip("|").split("|")])
    head = [h.lower() for h in rows[0]]
    body = [(r + [""] * len(head))[:len(head)] for r in rows[1:]]
    return pd.DataFrame(body, columns=head)

def read_returned(path):
    if path.suffix == ".md":
        return read_markdown(path)
    return pd.read_excel(path, sheet_name="Screening", dtype=str)

valid_ids = set(records["id"])
answers, summary = {}, []
for path in sorted(p for p in RAW.glob("*") if p.suffix in (".md", ".xlsx")):
    name = path.stem
    raw = read_returned(path).fillna("")
    for column in ANSWER:
        raw[column] = raw[column].astype(str).str.strip() if column in raw else ""
    raw["decision"] = raw["decision"].str.lower()
    raw["reason"] = raw["reason"].str.upper()
    raw = raw[raw["id"].str.lower() != "id"]                     # header rows repeated inside the file
    unknown = ~raw["id"].isin(valid_ids)
    raw = raw[~unknown]
    repeated = int(raw["id"].duplicated().sum())
    distinct = raw.drop_duplicates(["id", "decision", "reason"])
    conflict = distinct["id"][distinct["id"].duplicated()].unique()   # same record, different answers
    raw = raw.drop_duplicates("id")
    raw.loc[raw["id"].isin(conflict), ["decision", "reason"]] = ""
    bad_decision = ~raw["decision"].isin(DECISIONS + [""])
    bad_reason = (raw["decision"] == "exclude") & ~raw["reason"].isin(REASONS)
    raw.loc[bad_decision, "decision"] = ""
    raw.loc[raw["decision"] != "exclude", "reason"] = ""
    clean = records[READ].merge(raw[["id"] + ANSWER], on="id", how="left").fillna("")
    write_sheet(clean, RETURNED / f"screening-sheet-{name}.xlsx")
    answers[name] = clean.set_index("id")[ANSWER]
    counts = clean["decision"].value_counts()
    summary.append({"reviewer": name, "format": path.suffix[1:], "rows read": int(len(raw) + repeated + unknown.sum()),
                    "unknown id": int(unknown.sum()), "repeated rows": repeated,
                    "conflicting": len(conflict), "invalid decision": int(bad_decision.sum()),
                    "exclude without reason": int(bad_reason.sum()), "decided": int((clean["decision"] != "").sum()),
                    **{d: int(counts.get(d, 0)) for d in DECISIONS}})
summary = pd.DataFrame(summary)
table(summary, bar="decided")

reviewer,format,rows read,unknown id,repeated rows,conflicting,invalid decision,exclude without reason,decided,include,exclude,doubt
chatgpt,xlsx,2734,0,0,0,0,0,2734,1288,616,830
claude,md,102,0,0,0,0,0,102,7,76,19
copilot,md,2734,0,0,0,0,0,0,0,0,0
deepseek,md,3731,0,2317,2,0,0,1412,296,1115,1
grok,xlsx,2734,0,0,0,0,0,2734,563,2171,0


All the decisions in one file, one row per record and reviewer, for the comparisons that follow and for T0.3.

In [5]:
long = pd.concat([a.assign(reviewer=name).reset_index() for name, a in answers.items()], ignore_index=True)
long = long[long["decision"] != ""][["id", "reviewer"] + ANSWER].sort_values(["id", "reviewer"])
long.to_csv(DECISIONS_ALL, index=False, encoding="utf-8")
print(len(long), "decisions written to", DECISIONS_ALL)

6982 decisions written to reviewer_decisions.csv


## 4. Agreement

Agreement between every pair of raters, the main reviewer included, over the records both have decided. As in the comparison with the first pass, a doubt counts as an inclusion, since both send the record to full text. It is given as a percentage and as Cohen's kappa; a pair with fewer than 30 records in common is not reported.

In [6]:
def kappa(a, b):
    """Cohen's kappa of two lists of binary labels."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    observed = (a == b).mean()
    expected = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())
    return float("nan") if expected == 1 else (observed - expected) / (1 - expected)

main = pd.read_csv(SCREENING, dtype=str).fillna("").set_index("id")[ANSWER]
raters = {"main": main, **answers}
decided = {name: a.index[a["decision"] != ""] for name, a in raters.items()}
print("decided by the main reviewer:", len(decided["main"]))

rows = []
for x, y in combinations(raters, 2):
    both = decided[x].intersection(decided[y])
    if len(both) < 30:
        continue
    a = raters[x].loc[both, "decision"] != "exclude"
    b = raters[y].loc[both, "decision"] != "exclude"
    rows.append({"rater_a": x, "rater_b": y, "records": len(both),
                 "agreement": round(float((a == b).mean()), 3), "kappa": round(kappa(a, b), 3),
                 "a_to_full_text": round(float(a.mean()), 3), "b_to_full_text": round(float(b.mean()), 3)})
agreement = pd.DataFrame(rows)
agreement.to_csv(AGREEMENT, index=False, encoding="utf-8")
table(agreement, bar="kappa")

decided by the main reviewer: 2734


rater_a,rater_b,records,agreement,kappa,a_to_full_text,b_to_full_text
main,chatgpt,2734,0.465000,0.137000,0.281000,0.775000
main,claude,102,1.000000,1.000000,0.255000,0.255000
main,deepseek,1412,0.821000,0.519000,0.280000,0.210000
main,grok,2734,0.789000,0.431000,0.281000,0.206000
chatgpt,claude,102,0.451000,0.128000,0.765000,0.255000
chatgpt,deepseek,1412,0.417000,0.107000,0.763000,0.210000
chatgpt,grok,2734,0.419000,0.121000,0.775000,0.206000
claude,deepseek,57,0.737000,-0.092000,0.211000,0.053000
claude,grok,102,0.725000,0.122000,0.255000,0.118000
deepseek,grok,1412,0.798000,0.392000,0.210000,0.210000


Records that every reviewer with a complete file sends to full text, and records on which they split. These are the first ones to read in the human review.

In [7]:
complete = [n for n in answers if (answers[n]["decision"] != "").all()]
votes = pd.DataFrame({n: answers[n]["decision"] != "exclude" for n in complete})
print("reviewers with a complete file:", ", ".join(complete))
split = votes.any(axis=1) & ~votes.all(axis=1)
print("all send to full text:", int(votes.all(axis=1).sum()), "| all exclude:", int((~votes).all(axis=1).sum()),
      "| split:", int(split.sum()))

reviewers with a complete file: chatgpt, grok
all send to full text: 546 | all exclude: 599 | split: 1589


Check against the control set: these works must reach the full text, so a reviewer who excludes them is applying the criteria too strictly. K25–K27 test the safety coverage criterion (I2b).

In [8]:
control = pd.read_csv("../T0.1-protocol/control_set.csv", dtype=str).fillna("")
by_doi = corpus[corpus["doi"] != ""].set_index("doi")["id"].to_dict()
control["record"] = control["doi"].str.lower().map(by_doi)
control = control.dropna(subset=["record"]).set_index("record")[["id", "reference"]]
for name, a in answers.items():
    control[name] = a["decision"].reindex(control.index)
excluded = {n: int((control[n] == "exclude").sum()) for n in answers}
print("control works in the corpus:", len(control), "| excluded by:", excluded)
table(control[(control[list(answers)] == "exclude").any(axis=1)].reset_index())

control works in the corpus: 26 | excluded by: {'chatgpt': 2, 'claude': 0, 'copilot': 0, 'deepseek': 4, 'grok': 6}


record,id,reference,chatgpt,claude,copilot,deepseek,grok
R2116,K04,"Tadeusiewicz and Hałgas, JETTA 2017",exclude,,,include,exclude
R1271,K09,"Grzechca, ECCTD 2011",include,,,exclude,include
R1853,K12,"Ankush et al. (CODA), ETS 2026",include,,,exclude,include
R0066,K14,"Lin and Elcherif, IJCTA 1985",include,,,,exclude
R0894,K23,"Biswas, Blanton and Stratigopoulos, DATE 2005",exclude,,,,exclude
R1706,K25,"Pavlidis et al. (SymBIST), TCAS-I 2021",include,,,exclude,exclude
R2224,K26,"Sunter et al., ITC 2020",include,,,,exclude
R1880,K27,"Praselski and Sunter, ETS 2026",include,,,exclude,exclude


## Conclusions and notes

- **Sheet:** `screening-sheet.xlsx` holds the 2734 records of the corpus in a fixed random order, with only the title, year, source and abstract, and blank `decision`, `reason` and `notes` columns with drop-down lists for the codes.
- **Returned files: five, all from AI assistants, in two formats.** They are kept as returned in `reviewers/raw/` and rewritten in the format of the sheet as `reviewers/screening-sheet-<name>.xlsx`.
  - *chatgpt* and *grok* decided all 2734 records.
  - *deepseek* decided 1412 records, about half of the corpus. Its file is a series of 15 tables, one after another, that repeat many records (2317 repeated rows); 2 records with different answers in different tables are left undecided.
  - *copilot* returned the list of records with no decision at all.
  - *claude* decided only the 102 records of block S2 that the main reviewer had not screened, as a first approximation: 7 include, 19 doubt, 76 exclude. It sends 25 % of them to full text, between grok (12 %) and chatgpt (77 %), and excludes none of the control works.
- **The assistants do not agree with each other.** chatgpt and grok, the two complete files, agree on 42 % of the records (kappa 0.12). chatgpt sends 77 % of the corpus to full text and grok 21 %; they agree on 546 records to read and 599 to exclude, and split on 1589.
- **Control set.** chatgpt excludes 2 of the 26 control works in the corpus (K04 and K23); grok excludes 6, among them the three works of the safety block (K25–K27), so it is not applying criterion I2b; deepseek excludes 4 of the 15 it decided (K09, K12, K25 and K27).
- **deepseek and grok are closer to each other** (80 % agreement, kappa 0.39 on 1412 records) than either is to chatgpt (kappa 0.11 and 0.12), but both exclude control works.
- **Against the main reviewer** (2632 records decided; the 102 of block S2 are pending): deepseek agrees on 82 % (kappa 0.54, on 1355 records), grok on 79 % (kappa 0.44) and chatgpt on 47 % (kappa 0.14). The main reviewer sends 28 % of the records to full text, close to grok and deepseek (21–22 %) and far from chatgpt (78 %).
- **Records to look at again:** 165 that the main reviewer excludes and both chatgpt and grok send to full text. They are the cheapest check against a lost study.

### Notes

- These are not independent human reviewers. With a kappa of 0.12 between the two complete files, none of them can stand in for the human screening or be used as the second reviewer of the protocol. At most they are a first pass, as `screening_ai.csv` was, and the protocol has to say so.
- The spreadsheets are not byte-identical between runs (the zip archive stores the time it is written), but their content is.